In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pyreadstat
import json

In [ ]:
# Lets create a global configuration block sourced from: ODiN2023_Codeboek_v1_0 which we can use during the entire analysis

# --- Geographic filter ---
amsterdam_gem_code = 363      # CBS municipality code for Amsterdam (WoGem)

# --- Trip purpose filter (MotiefV) ---
# Source: ODiN2023 codebook, variable MotiefV
commute_code_amst = 1.0    # 1 = "Van en naar het werk" (commuting)
# Excluded: 2 = "Zakelijk bezoek in werksfeer" (business travel)
# Reason: business trips are often employer-reimbursed, biasing cost coefficients

# --- Weekday filter (Weekdag) ---
# Source: ODiN2023 codebook, variable Weekdag
# 1=Sunday, 2=Monday, 3=Tuesday, 4=Wednesday, 5=Thursday, 6=Friday, 7=Saturday
weekday_codes_amst = [2.0, 3.0, 4.0, 5.0, 6.0]   # Monday to Friday

# ---- Age bands (matching NTS Age_B04ID 9-category scheme) ----
age_bins = [0, 5, 11, 17, 21, 30, 40, 50, 60, 200]
age_labels = [1, 2, 3, 4, 5, 6, 7, 8, 9]

# Income: Amsterdam decile transfered to quintiles ----
decile_to_quintile= {
    1: 1, 2: 1,   # Q1 (lowest)
    3: 2, 4: 2,   # Q2
    5: 3, 6: 3,   # Q3
    7: 4, 8: 4,   # Q4
    9: 5, 10: 5,  # Q5 (highest)
}

# --- KHvm: Main mode class codes (used for choice set definition) ---
# Source: ODiN2023 codebook, variable KHvm
maincode_car_driver = 1.0    # Personenauto - bestuurder → alternative: CAR
maincode_pt         = 4.0    # Bus/tram/metro → alternative: PUBLIC TRANSPORT
maincode_bike       = 5.0    # Fiets (e-bike + regular) → alternative: BICYCLE
maincode_walk       = 6.0    # Te voet → EXCLUDED (not a motorised mode)
maincode_other      = 7.0    # Overig → EXCLUDED

# Choice set: the three alternatives in our MNL model
choice_set = [maincode_car_driver, maincode_pt, maincode_bike]

# Hvm: Detailed mode codes (used for BTM analysis only (Bus, Tram, Metro)) ---
# Source: ODiN2023 codebook, variable Hvm
dtcode_car   = 1.0    # Personenauto
dtcode_train = 2.0    # Trein
dtcode_bus   = 3.0    # Bus
dtcode_tram  = 4.0    # Tram
dtcode_metro = 5.0    # Metro
dtcode_ebike = 7.0    # Elektrische fiets
dtcode_bike  = 8.0    # Niet-elektrische fiets
dtcode_walk  = 9.0    # Te voet

# BTM group: detailed modes that map to KHvm==4
btm_codes = [dtcode_bus, dtcode_tram, dtcode_metro]

# --- Distance conversion ---
hm_to_km = 0.1    # AfstV is recorded in hectometers

# --- Cleaning thresholds ---
min_trip_dur = 2 # in mins; below 2mins likely data entry error
max_trip_dur = 120 # in mins; above 120 mins is outside typical Amsterdam urban commute
min_trip_distance = 0.2 # in km; below is unlikely for any of our three modes
max_trip_distance = 50 # in km; above is outside Amsterdam urban area

# --- PT-specific attributes ---
# Transfer penalty estimation (literature values for imputation)
# Source: Wardman & Hine (2000), "Costs of Interchange: A Review of the Literature"
avg_transfer_time_min = 8    # median platform-to-platform transfer time
transfer_penalty_min  = 5    # psychological penalty per transfer (equiv. in-vehicle min)

# --- Peak hour definition ---
am_peak_start = 7 
am_peak_end   = 9
pm_peak_start = 17
pm_peak_end   = 19

modes_class_codes = {
    1.0: "Car - driver", 
    2.0: "Car - passenger", 
    3.0: "Train",
    4.0: "Bus/Tram/Metro (BTM)",
    5.0: "Bicycle",
    6.0: "Walking",
    7.0: "Other"
}

# --- Detailed mode distribution (Hvm) ---
detailed_modes_codes = {
    1.0: "Car", 2.0: "Train", 3.0: "Bus", 4.0: "Tram",
    5.0: "Metro", 6.0: "Speedpedelec", 7.0: "E-bike",
    8.0: "Bicycle", 9.0: "Walking", 10.0: "Coach",
    11.0: "Van", 14.0: "Taxi", 16.0: "Motorcycle",
    17.0: "Moped (45km/h)", 18.0: "Moped (25km/h)"
}


In [ ]:
# Read ODiN 2023 trip-level data
odin_data, meta = pyreadstat.read_sav("project_root\data\raw\amsterdam\ODiN2023_Databestand.sav") #update the path accordingly to load the data   
columns = meta.column_labels

In [ ]:
#update the path accordingly to load the columns english names    
with open(r"project_root\data\data_documentation\english_column_labels.txt", 'r', encoding='utf-8') as file:
    english_names = file.read().splitlines()

In [ ]:
#update the path accordingly to load the column mapping   

meta.column_labels_en = list(english_names)

column_mapping = dict(zip(odin_data.columns, english_names)) # Dutch to English Mapping

with open(r"project_root\data\data_documentation\column_mapping.json", 'w', encoding='utf-8') as file:
    json.dump(column_mapping, file, ensure_ascii=False, indent=4)

In [ ]:
key_vars = {
    "MotiefV"  : "Trip purpose",
    "KHvm"     : "Class transport mode",
    "Hvm"      : "Main transport mode ",
    "WoGem"    : "Residential Municipality",
    "Weekdag"  : "Day of Week",
    "Feestdag" : "Public holiday flag",
}

n = len(odin_data) 

for col, label in key_vars.items():
    print(f"\n{'='*30}\n {label} [{col}]\n{'='*30}") 
    
    stats = odin_data[col].value_counts(dropna = False, normalize = True).sort_index() #we normalize to get the proportions 
    result = pd.DataFrame({
        "count": (stats * n).astype(int), 
        "pct (%)": (stats * 100).round(2) #
    })
    print(result)

In [ ]:
codebook = pd.read_csv(r"data\data_documentation\ODiN2023_Codeboek_v1.0.tab", sep = "\t", header = None, dtype = str, encoding = "latin-1")

In [ ]:
codebook = codebook.fillna("").apply(lambda col: col.str.strip().str.strip())

In [ ]:
codelabels = {}
current_var = None

for _, row in codebook.iterrows():
    var = row[0]
    code = row[3]
    desc = row[4]

    if var:
        current_var = var
        codelabels[current_var] = {}

    if current_var and code and code != "<missing>" and desc: 
        try:
            codelabels[current_var][float(code)] = desc
        except ValueError:
            codelabels[current_var][code] = desc

In [ ]:
#Save the stats results with the labels names (in Dutch) 
results = {}
n = len(odin_data)

for col, label in key_vars.items():
    stats = odin_data[col].value_counts(dropna = False, normalize = True).sort_index() #we normalize to get the proportions
    var_labels = codelabels.get(col, {})
    index_labels = [
        var_labels.get(x, str(int(x)) if x == x else "NaN") if x == x else "NaN"
        for x in stats.index
        ]
    results[col] = pd.DataFrame({
        "label": index_labels,
        "pct" : (stats.values * 100). round(2), #tburn the shares into %
        "count":(stats.values * n),
    })

In [ ]:
# Obtain Data
plot_motive = results["MotiefV"].sort_values('count', ascending=False)

# Def the plot
plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=plot_motive, 
    x='count', 
    y='label'
)

# --- Add percentages to the right of each bar ---
for i, p in enumerate(ax.patches):
    # Get the percentage from your 'pct' column
    percentage = plot_motive['pct'].iloc[i]
    
    # Calculate position
    x_pos = p.get_width() # The end of the bar
    y_pos = p.get_y() + p.get_height() / 2 # The middle of the bar's thickness
    
    # Annotate
    ax.annotate(f'{percentage:.1f}%', 
                (x_pos, y_pos), 
                va='center', 
                ha='left', 
                xytext=(8, 0),             # Push text 8 points to the right
                textcoords='offset points',
                fontsize=10)

# Labels
plt.title('Absolute Counts by Travel Purpose', fontsize=14)
plt.xlabel('Number of Trips', fontsize=12)
plt.ylabel('Motive', fontsize=12)

# Increase x-limit to make room for the text labels
plt.xlim(0, plot_motive['count'].max() * 1.15)

plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
rename_map = {
    # Identifiers
    "OP" : "new_id", #0- no ; 1= yes
    "OPID" : "person_id",
    "VerplID" : "trip_id",
    "Verpl" : "new_trip_id", # 0 = no, 1 = new trip ; 6 = serries ; 7 = freight trip

    # Geography
    "WoGem" : "home_municipality",
    "VertGem" : "origin_municipality",
    "AankGem" : "dest_municipality",
    "VertPC" : "origin_postcode",
    "AankPC" : "dest_postcode",
    "BuurtAdam" : "amsterdam_neighborhood",

    # Trip characteristics
    "MotiefV" : "trip_purpose", # 1 = commuting ("Van en naar het werk")
    "Hvm" : "mode_detailed", # granular mode code
    "KHvm" : "mode_class", # 1 = car , 4 = PT, 5 = bike
    "HvmRol" : "mode_role", # 1=driver  2 =passenger
    "AantRit" : "n_legs", #number of legs - transfers in public transport
    "Maand" : "month",
    "Weekdag" : "weekday", # 2=Mon ... 6=Fri
    "Feestdag" : "is_holiday",
    "VertUur" : "departure_hour",
    "VertMin": "departure_minute",
    "Reisduur" : "travel_time_min",       # total door-to-door minutes
    "AfstV" : "distance_hm",           # trip distnace in hectometers — convert later imto km

    # Person characteristics
    "Leeftijd" : "age",
    "Geslacht" : "gender",                # 1=male, 2=female
    "HHBestInkG" : "income_decile",         # 1 (lowest) to 10 (highest)
    "OPRijbewijsAu" : "has_driving_license",   # 1=yes, 2=no
    "HHAuto" : "n_cars_household",

    # Survey weights
    "FactorV" : "weight_trip", #survey trip weight
    "FactorP" : "weight_person", #survey person weight
}

# Select and rename
work_data = odin_data[rename_map.keys()].rename(columns = rename_map).copy()

In [ ]:
work_data.head(5)

In [ ]:
#convert distance in hectometers to km
hm_to_km = 0.1   # converting coefficient
work_data["distance_km"] = (work_data.distance_hm * hm_to_km).round(3)

# Count the number of trnasfers from the number of legs in a trip
# For non-PT modes, n_legs is always 1 (no legs), so transfers = 0
work_data["n_transfers"] = np.where(work_data.mode_class == maincode_pt, np.maximum(work_data.n_legs.fillna(1.0) - 1, 0),0).astype(np.int8)

# Transfer flag indicator for PT for trips with at least one transfer
work_data["has_transfer"] = (work_data["n_transfers"] > 0).astype(np.int8)

In [ ]:
#Lets insert a flag when a trip was during peak hours
#AM peak: 07:00h - 09:00h ; 
#PM peak: 17:00h - 19:00h (selected based on the peak definition from config file)

work_data["is_peak"] = (
    ((work_data.departure_hour.values >= am_peak_start) & (work_data.departure_hour.values < am_peak_end)) |
    ((work_data.departure_hour.values >= pm_peak_start) & (work_data.departure_hour.values < pm_peak_end))
).astype(np.int8)

In [ ]:
# Maps KHvm codes to analytical labels used in all downstream steps
mode_label_map = {
    maincode_car_driver : "car",
    maincode_pt : "pt", #public transport
    maincode_bike : "bike",
}

work_data["chosen_mode"] = (work_data.mode_class.map(mode_label_map).astype("category"))

In [ ]:
# Look for all Amsterdam Trips
base_mask = (
    (work_data.home_municipality == amsterdam_gem_code) &
    (work_data.trip_purpose == commute_code_amst) &
    (work_data.weekday.isin(weekday_codes_amst)) &
    (work_data.is_holiday == 0)
    ) 

base_amst_data = work_data[base_mask].copy()
print(f"Amsterdam commuting trips (all modes): {len(base_amst_data):,}")

In [ ]:
#  Detailed mode distribution (Hvm) in Amsterdam
dtl_mode_counts = base_amst_data.mode_detailed.value_counts().sort_index() # count trips per detailed transport mode class 
dtl_mode_pct    = (dtl_mode_counts / len(base_amst_data) * 100).round(1) #count the pct for each detailed transport mode class
dtl_mode_table  = pd.DataFrame({
    "Mode (Detailed)": dtl_mode_counts.index.map(lambda x: detailed_modes_codes.get(x, f"Other ({x})")),
    "Count": dtl_mode_counts.values,
    "Share(%)": dtl_mode_pct.values
})

print("Detailed transport mode - Amsterdam commuting")
print(dtl_mode_table.to_string(index=False))
print()

In [ ]:
# --- Detailed mode distribution in Amsterdam---
plot_dtl = dtl_mode_table.sort_values("Share(%)", ascending=False)

plt.figure(figsize=(12, 8))
ax = sns.barplot(
    data=plot_dtl,
    x="Mode (Detailed)",  # Categorical on X for vertical
    y="Share(%)",         # Numerical on Y for vertical
    hue="Mode (Detailed)",
    legend=False
)

# Add percentages on top of bars
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', 
                xytext=(0, 9), 
                textcoords='offset points',
                fontsize=10)

plt.title('Detailed Transport Mode Distribution: Amsterdam Commuting', fontsize=15, pad=20)
plt.xlabel('Transport Mode', fontsize=12)
plt.ylabel('Share of Trips (%)', fontsize=12)

# 1. Increase height to prevent labels being cut off
plt.ylim(0, plot_dtl["Share(%)"].max() * 1.2) 

# 2. Rotate labels so they are readable
plt.xticks(rotation=45, ha='right')

# 3. Add horizontal gridlines for easier reading
plt.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
# --- Mode class distribution (KHvm) ---
cls_mode_counts = base_amst_data.mode_class.value_counts().sort_index() # count trips per transport mode class 
cls_mode_pct    = (cls_mode_counts / len(base_amst_data) * 100).round(1) #count the pct for each transport mode class
cls_mode_table  = pd.DataFrame({
    "Mode Class": cls_mode_counts.index.map(lambda x: modes_class_codes.get(x, f"Other ({x})")),
    "Count": cls_mode_counts.values,
    "Share(%)": cls_mode_pct.values
})
print("=== Detailed transport mode — Amsterdam commuting ===")
print(cls_mode_table.to_string(index=False))

print()

In [ ]:
# Sort the data
plot_cls = cls_mode_table.sort_values("Share(%)", ascending=False)

plt.figure(figsize=(10, 7)) # Slightly narrower usually works better for fewer classes
ax = sns.barplot(
    data=plot_cls,
    x="Mode Class",     # Category on X
    y="Share(%)",       # Value on Y
    hue="Mode Class",
    legend=False
)

# Add percentages on top of each bar
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', 
                xytext=(0, 9), 
                textcoords='offset points',
                fontsize=11,
                fontweight='bold')

# Labels and Title
plt.title('Transport Mode Class Distribution: Amsterdam Commuting', fontsize=15, pad=20)
plt.xlabel('Transport Mode Class', fontsize=12)
plt.ylabel('Share of Trips (%)', fontsize=12)

# Increase the Y-limit to make room for the labels at the top
plt.ylim(0, plot_cls["Share(%)"].max() * 1.2)

# Optional: Clean up the look
plt.xticks(rotation=0) # Only rotate if the class names are very long
plt.grid(axis='y', linestyle='--', alpha=0.5)
sns.despine() # Removes the top and right border for a cleaner look

plt.tight_layout()
plt.show()


In [ ]:
# Filter to BTM trips only (KHvm == 4)
btm_amst_data = base_amst_data[base_amst_data.mode_class == 4]

btm_detail = btm_amst_data.mode_detailed.value_counts().sort_index()
btm_pct = (btm_detail / len(btm_amst_data) * 100).round(1)

btm_table = pd.DataFrame({
    "Detailed mode": btm_detail.index.map(lambda x: detailed_modes_codes.get(x, f"Code {x}")),
    "Count": btm_detail.values,
    "Share within PT (%)": btm_pct.values
})

print("Composition of BTM code (Public Transport)")
print(f"Total BTM trips: {len(btm_amst_data):,}")
print()
print(btm_table.to_string(index=False))
print()
print("Decision: bus + tram + metro treated as one PT alternative.")

In [ ]:
#Lets define a function for easier filtering and keeping a log of the changes we are making of the dataset
transformation_log = []

def apply_filter(dataset, mask, description):
    """Apply a boolean mask, log the result, return filtered df."""
    n_before = len(dataset)
    dataset_r = dataset[mask].copy()
    n_after  = len(dataset_r)
    n_dropped = n_before - n_after
    entry = f"{description}: {n_before:,} → {n_after:,} rows (dropped {n_dropped:,})"
    transformation_log.append(entry)
    print(entry)
    return dataset_r

# Start from full raw dataset
amst_data = work_data.copy()

# Filter 1: regular trips only (Verpl == 1, excludes trip series and freight)
amst_data = apply_filter(amst_data, amst_data.new_trip_id == 1.0,
                         "F1: Regular trips only (new_trip_id == 1)")

# Filter 2: Amsterdam residential municipality
amst_data = apply_filter(amst_data, amst_data.home_municipality == amsterdam_gem_code,
                         "F2: Amsterdam residents (home_municipality == {amsterdam_gem_code})")

# Filter 3: commuting purpose only
amst_data = apply_filter(amst_data, amst_data.trip_purpose == commute_code_amst,
                         "F3: Commuting trips (trip_purpose == 1)")

# Filter 4: weekdays only (Mon-Fri)
amst_data = apply_filter(amst_data, amst_data.weekday.isin(weekday_codes_amst),
                         "F4: Working days only (Mon-Fri)")

# Filter 5: exclude public holidays
amst_data = apply_filter(amst_data, amst_data.is_holiday == 0.0,
                         "F5: Exclude public holidays (is_holiday == 0)")

# Filter 6: our three modes only (car driver, PT, bicycle)
amst_data = apply_filter(amst_data, amst_data.mode_class.isin(choice_set),
                         "F6: Choice set modes only (car driver, PT, bicycle)")

# Filter 7: car driver role only (HvmRol == 1 for car, or not applicable)
# Exclude car passengers who were classified as car driver in KHvm (edge cases)
amst_data = apply_filter( amst_data, ~((amst_data.mode_class == maincode_car_driver) & (amst_data.mode_role == 2.0)),
                          "F7: Exclude car passengers misclassified as drivers (mode_role == 2)"
)

# Filter 8: Unknown income ( income_decile == 11)
# Reason: income decile used in robustness checks; unknown code unusable
amst_data = apply_filter( amst_data, amst_data.income_decile != 11.0,
                          "F8: Unknown income decile (income_decile == 11)" )

# Filter9: Unknown car ownership (n_cars_household == 10)
amst_data = apply_filter( amst_data, amst_data.n_cars_household != 10.0,
                          "F9: Unknown car ownership (n_cars_household == 10)")

# Filter 10: Unknown driving licence status (has_driving_license == 2)
amst_data = apply_filter( amst_data, amst_data.has_driving_license != 2.0,
                          "F10: Unknown driving licence status (has_driving_license == 2)")

# Filter 11: Min trip duration 
amst_data = apply_filter( amst_data, ((amst_data.travel_time_min >= min_trip_dur) & (amst_data.travel_time_min <= max_trip_dur)), 
                          "F11: Set max duration trip treshold")

#Filter 12: 
amst_data = apply_filter( amst_data, ((amst_data.distance_km >= min_trip_distance) & (amst_data.distance_km <= max_trip_distance)), 
                          "F12: Set max duration trip treshold")

In [ ]:
amst_data["city"] = "Amsterdam"

In [ ]:
# Bin Amsterdam raw age (Leeftijd) into NTS Age_B04ID 9-band scheme
# Must run BEFORE merging Amsterdam + London merge in phase 3
amst_data["age_band"] = pd.cut(amst_data.age, bins = age_bins, labels = age_labels, right = False).astype(int)

# Collapse Amsterdam income deciles → quintiles
amst_data["income_quintile"] = amst_data.income_decile.map(decile_to_quintile)

In [ ]:
mode_counts = amst_data.chosen_mode.value_counts().sort_index()

In [ ]:
mode_counts

In [ ]:
variables  = ["travel_time_min", "distance_km"]
mode_colors = {"car": "#2196F3", "pt": "#FF9800", "bike": "#4CAF50"}
transport_modes = ["car", "pt", "bike"]
mode_labels = {"car": "Car (driver)", "pt": "Public Transport", "bike": "Bicycle"}

fig, axes = plt.subplots(2,1, figsize = (8, 8),sharex = False ,sharey = True)

for ax, var in zip(axes, variables):
    for mode in transport_modes:
        subset = amst_data[amst_data.chosen_mode == mode]
        # Weighted histogram using survey weights
        ax.hist(
            subset[var],
            weights = subset.weight_trip,
            bins = 30,
            alpha = 0.55,
            color = mode_colors[mode],
            label = mode_labels[mode],
            edgecolor = "none"
        )
    ax.set_xlabel("Travel time (minutes)" if var == "travel_time_min" else "Distance (km)", fontsize=10)
    ax.set_ylabel("Weighted frequency", fontsize = 10)
    ax.set_title("Travel time distribution by mode" if var == "travel_time_min" else "Trip distance distribution by mode",
                 fontsize = 11, fontweight = "bold")
    ax.legend(fontsize = 9)
    ax.grid(axis = "y", alpha = 0.3, linewidth = 0.5)
    ax.spines[["top", "right"]].set_visible(False)

plt.subplots_adjust(hspace = 0.4)
# plt.tight_layout()
# plt.savefig("data/raw/odin/path", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 1. Melt the data so variables are in a single column for grouping
amst_melted = amst_data.melt(id_vars = ['chosen_mode'], value_vars = variables)

In [ ]:
# 2. Calculate Q1 and Q3 per group
amst_stats = amst_melted.groupby(['variable', 'chosen_mode'],observed = False).value.quantile([0.25, 0.75]).unstack()
amst_stats.columns = ['Q1', 'Q3']

In [ ]:
# 3. Define the upper and lower boundaries (fences)
amst_stats['iqr'] = amst_stats.Q3 - amst_stats.Q1 
amst_stats['lower'] = amst_stats.Q3 - 1.5 * amst_stats.iqr
amst_stats['upper'] = amst_stats.Q3 + 1.5 * amst_stats.iqr

In [ ]:
# 4. Count outliers by merging stats back to the melted data
amst_merged = amst_melted.merge(amst_stats, on = ['variable', 'chosen_mode'])
amst_merged['is_outlier'] = (amst_merged.value < amst_merged.lower) | (amst_merged.value > amst_merged.upper)
n_outliers = amst_merged.groupby(['variable', 'chosen_mode'], observed = False).is_outlier.sum().reset_index()

In [ ]:
# 5. Build final report
amst_report = amst_stats.reset_index().merge(n_outliers, on = ['variable', 'chosen_mode'])
amst_report['mode'] = amst_report.chosen_mode.map(mode_labels)

In [ ]:
amst_report

In [ ]:
# 1. Group by mode and calculate both metrics at once
stats = amst_data.groupby("chosen_mode",observed = False).weight_trip.agg(["count", "sum"])

# 2. Vectorized calculation of percentages for the whole table
total_count = len(amst_data)
total_weight = amst_data.weight_trip.sum()

stats["unwt_pct"] = (stats['count'] / total_count) * 100
stats["wt_pct"]   = (stats['sum'] / total_weight) * 100

#Save the results per transport mode
results_dict = {
    mode_labels[mode]: [row['unwt_pct'], row['wt_pct']] for mode, row in stats.iterrows()
}

report_pct = pd.DataFrame.from_dict(
    results_dict, 
    orient = 'index', 
    columns = ['Unweighted Score %', 'Weighted Score %']
)

In [ ]:
# 1. Map the labels and select only the percentage columns
plot_pct = stats[['unwt_pct', 'wt_pct']].copy()
plot_pct.index = plot_pct.index.map(mode_labels)
plot_pct.index.name = 'Transport Mode'

# 2. Reshape (melt) the data for side-by-side plotting
plot_pct = plot_pct.reset_index().melt(id_vars = 'Transport Mode', var_name= 'Type', value_name='Share (%)')
plot_pct['Type'] = plot_pct['Type'].map({'unwt_pct': 'Unweighted', 'wt_pct': 'Weighted'})


plt.figure(figsize=(8, 5))
sns.barplot(data = plot_pct, x = 'Transport Mode', y = 'Share (%)', hue = 'Type', palette='viridis')


plt.title('Mode Share Comparison: Weighted vs. Unweighted', fontsize= 14, pad = 15)
plt.ylabel('Share of Total Trips (%)')
# plt.xlabel('')
plt.legend(title = 'Calculation Type')
sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
# --- Export to a directory ---
amst_data.to_csv(r"project_root\data\processed\amst_processed.csv", index = False)